In [4]:
# --- 1. RAPPORT D'AUDIT : Transactions ---
df_trans = pd.read_csv('transactions.csv')
total_t = len(df_trans)

# Conversion des dates pour les contrôles logiques
df_trans['purchase_date'] = pd.to_datetime(df_trans['purchase_date'], errors='coerce')
df_trans['actual_delivery_date'] = pd.to_datetime(df_trans['actual_delivery_date'], errors='coerce')

# Détection des anomalies selon le catalogue [cite: 503]
mask_t_id = ~df_trans['transaction_id'].astype(str).str.startswith('TRX_')
mask_t_logic = (df_trans['actual_delivery_date'] < df_trans['purchase_date'])
mask_t_val = (df_trans['quantity'] <= 0) | (df_trans['total_price'] < 0)

# Outliers de prix (Z-score > 3) pour ne pas fausser le panier moyen [cite: 310-311]
z_scores = (df_trans['total_price'] - df_trans['total_price'].mean()) / df_trans['total_price'].std()
mask_t_out = np.abs(z_scores) > 3

print(f"--- Rapport d'Audit pour Transactions ---")
print(f"❌ Identifiants corrompus (BADID)      : {mask_t_id.sum()} ({mask_t_id.sum()/total_t*100:.1f}%)")
print(f"📅 Dates de livraison incohérentes     : {mask_t_logic.sum()} ({mask_t_logic.sum()/total_t*100:.1f}%)")
print(f"💰 Prix ou quantités négatifs/nuls     : {mask_t_val.sum()} ({mask_t_val.sum()/total_t*100:.1f}%)")
print(f"📈 Outliers de prix (Z-score > 3)      : {mask_t_out.sum()} ({mask_t_out.sum()/total_t*100:.1f}%)")

# Nettoyage Silver
df_t_silver = df_trans[~(mask_t_id | mask_t_logic | mask_t_val)].copy()
median_p = df_t_silver['total_price'].median()
df_t_silver.loc[mask_t_out, 'total_price'] = median_p

print("-" * 30)
print(f"✅ Nettoyage fini : {len(df_t_silver)} lignes valides sur {total_t} soit {len(df_t_silver)/total_t*100:.1f}%")
df_t_silver.to_csv('transactions_silver.csv', index=False)

--- Rapport d'Audit pour Transactions ---
❌ Identifiants corrompus (BADID)      : 67164 (3.4%)
📅 Dates de livraison incohérentes     : 66726 (3.3%)
💰 Prix ou quantités négatifs/nuls     : 121889 (6.1%)
📈 Outliers de prix (Z-score > 3)      : 31064 (1.6%)
------------------------------
✅ Nettoyage fini : 1795492 lignes valides sur 2000000 soit 89.8%


In [5]:
# --- 2. RAPPORT D'AUDIT : Accounting ---
df_acc = pd.read_csv('accounting.csv')
total_a = len(df_acc)

# Détection des anomalies [cite: 555]
mask_a_id = ~df_acc['expense_id'].astype(str).str.startswith('ACC_')
mask_a_neg = df_acc['amount'] < 0
mask_a_fiscal = ~df_acc['fiscal_period'].str.contains(r'^\d{4}-Q[1-4]$', na=False)

print(f"📊 RAPPORT D'AUDIT : Accounting")
print(f"❌ Identifiants corrompus (BADID)      : {mask_a_id.sum()} ({mask_a_id.sum()/total_a*100:.1f}%)")
print(f"💰 Montants négatifs                   : {mask_a_neg.sum()} ({mask_a_neg.sum()/total_a*100:.1f}%)")
print(f"📅 Format de période invalide          : {mask_a_fiscal.sum()} ({mask_a_fiscal.sum()/total_a*100:.1f}%)")

# Nettoyage Silver (On garde les erreurs de période en 'Unknown' pour ne pas perdre la valeur financière) [cite: 226]
df_acc_silver = df_acc[~mask_a_neg].copy()
df_acc_silver.loc[mask_a_fiscal, 'fiscal_period'] = 'UNKNOWN_PERIOD'

print("-" * 30)
print(f"✅ Nettoyage terminé : {len(df_acc_silver)} entrées valides sur {total_a} soit {len(df_acc_silver)/total_a*100:.1f}%")
df_acc_silver.to_csv('accounting_silver.csv', index=False)

📊 RAPPORT D'AUDIT : Accounting
❌ Identifiants corrompus (BADID)      : 763 (5.1%)
💰 Montants négatifs                   : 748 (5.0%)
📅 Format de période invalide          : 761 (5.1%)
------------------------------
✅ Nettoyage terminé : 14252 entrées valides sur 15000 soit 95.0%


In [6]:
# --- 3. RAPPORT D'AUDIT : Carbon Regulations ---
df_carb = pd.read_csv('carbon_regulations.csv')
total_c = len(df_carb)

# Détection selon les contraintes du catalogue [cite: 592]
mask_c_id = ~df_carb['regulation_id'].astype(str).str.startswith('REG_')
mask_c_year = df_carb['year'] < 2020
mask_c_price = df_carb['co2_price_per_ton'] < 0

print(f"📊 RAPPORT D'AUDIT : Carbon Regulations")
print(f"❌ Identifiants corrompus (BADID)      : {mask_c_id.sum()} ({mask_c_id.sum()/total_c*100:.1f}%)")
print(f"📅 Années invalides (< 2020)           : {mask_c_year.sum()} ({mask_c_year.sum()/total_c*100:.1f}%)")
print(f"💰 Prix CO2 négatifs                   : {mask_c_price.sum()} ({mask_c_price.sum()/total_c*100:.1f}%)")

# Nettoyage Silver
df_carb_silver = df_carb[~(mask_c_year | mask_c_price | mask_c_id)].copy()

print("-" * 30)
print(f"✅ Nettoyage terminé : {len(df_carb_silver)} régulations valides sur {total_c} soit {len(df_carb_silver)/total_c*100:.1f}%")
df_carb_silver.to_csv('carbon_regulations_silver.csv', index=False)

📊 RAPPORT D'AUDIT : Carbon Regulations
❌ Identifiants corrompus (BADID)      : 0 (0.0%)
📅 Années invalides (< 2020)           : 0 (0.0%)
💰 Prix CO2 négatifs                   : 0 (0.0%)
------------------------------
✅ Nettoyage terminé : 30 régulations valides sur 30 soit 100.0%


In [7]:
# --- 4. RAPPORT D'AUDIT : Sales Targets ---
df_tgt = pd.read_csv('sales_targets.csv')
total_tgt = len(df_tgt)

# Détection des anomalies [cite: 578]
mask_tgt_id = ~df_tgt['target_id'].astype(str).str.startswith('TGT_')
# Format YYYY-QN ou YYYY-MM
mask_tgt_period = ~df_tgt['period'].str.contains(r'^\d{4}-(Q[1-4]|0[1-9]|1[0-2])$', na=False)
# Correction de la variable df_targets -> df_tgt ici
mask_tgt_neg = (df_tgt['target_value'] < 0) | (df_tgt['achievement'] < 0)

print(f"📊 RAPPORT D'AUDIT : Sales Targets")
print(f"❌ Identifiants corrompus (BADID)      : {mask_tgt_id.sum()} ({mask_tgt_id.sum()/total_tgt*100:.1f}%)")
print(f"📅 Format de période invalide          : {mask_tgt_period.sum()} ({mask_tgt_period.sum()/total_tgt*100:.1f}%)")
print(f"💰 Valeurs cibles ou résultats négatifs: {mask_tgt_neg.sum()} ({mask_tgt_neg.sum()/total_tgt*100:.1f}%)")

# Nettoyage Silver
df_tgt_silver = df_tgt[~mask_tgt_neg].copy()

print("-" * 30)
print(f"✅ Nettoyage terminé : {len(df_tgt_silver)} objectifs valides sur {total_tgt} soit {len(df_tgt_silver)/total_tgt*100:.1f}%")
df_tgt_silver.to_csv('sales_targets_silver.csv', index=False)

📊 RAPPORT D'AUDIT : Sales Targets
❌ Identifiants corrompus (BADID)      : 167 (6.0%)
📅 Format de période invalide          : 170 (6.1%)
💰 Valeurs cibles ou résultats négatifs: 277 (9.9%)
------------------------------
✅ Nettoyage terminé : 2523 objectifs valides sur 2800 soit 90.1%


C:\Users\bouss\AppData\Local\Temp\ipykernel_35300\834197918.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_tgt_period = ~df_tgt['period'].str.contains(r'^\d{4}-(Q[1-4]|0[1-9]|1[0-2])$', na=False)
